# Tier 1 HCOs & their HCPs

## Version Control
| Version      | Description                                      |
|--------------|--------------------------------------------------|
|v1 (01/19)| HCO Level Patient Numbers |

## Step 1. Pulling Tx Counts, Dx Counts, All patient counts for the Tier 1 HCOs

In [0]:
-- ============================================
-- Create a temporary view containing ALL treated patients
-- Treatment is identified via:
--   1) Medical claims with relevant NDCs
--   2) Pharmacy claims with relevant NDCs (PAID only)
--   3) Medical claims with infusion / procedure codes
-- Date window applied AFTER union to standardize cohort timing
-- ============================================

CREATE OR REPLACE TEMPORARY VIEW mpsii_treatment_table AS

SELECT *
FROM (

    -- --------------------------------------------
    -- MEDICAL EVENTS: Treatment identified via NDC
    -- Uses Rendering NPI if available, else Referring NPI
    -- --------------------------------------------
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    -- --------------------------------------------
    -- PHARMACY EVENTS: Treatment identified via NDC
    -- Includes only PAID transactions to ensure true utilization
    -- Prescriber NPI used as treating provider
    -- --------------------------------------------
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE, -- Not applicable for pharmacy claims
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    -- --------------------------------------------
    -- MEDICAL EVENTS: Treatment identified via
    -- infusion / administration / transplant procedure codes
    -- --------------------------------------------
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366',
        'J1743','S9357','S9379',
        '38206','38230','38232',
        '38240','38241','38242',
        '38243','38250'
    )

)

-- --------------------------------------------
-- Apply treatment window filter
-- Ensures consistent cohort timing across all sources
-- --------------------------------------------
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';


In [0]:
-- =====================================================
-- Create a temporary view to calculate treated patients
-- at the HCO level (non-unique across time/events)
-- Provider attribution is rolled up from HCP → HCO
-- =====================================================

CREATE OR REPLACE TEMPORARY VIEW tx_claims_non_unique AS

-- ---------------------------------------------
-- Step 1: Attach HCO NPI to treatment claims
-- Mapping is done via HCP NPI → HCO NPI reference
-- ---------------------------------------------
WITH t1 AS (
    SELECT
        a.*,               -- All treatment claim attributes
        b.hco_npi          -- Parent HCO mapped from HCP
    FROM mpsii_treatment_table AS a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 AS b
        ON a.npi = b.hcp_npi
)

-- ---------------------------------------------
-- Step 2: Aggregate treated patients at HCO level
-- Counts DISTINCT patients treated at each HCO
-- ---------------------------------------------
SELECT
    hco_npi,
    COUNT(DISTINCT patient_id) AS tx_patients_non_unique
FROM t1

-- ---------------------------------------------
-- Filters:
-- 1) Exclude invalid / placeholder HCO NPIs
-- 2) Ensure provider attribution exists
-- ---------------------------------------------
WHERE hco_npi != '-'
  AND npi IS NOT NULL

-- ---------------------------------------------
-- Final aggregation & ranking
-- ---------------------------------------------
GROUP BY 1
ORDER BY 2 DESC;


In [0]:
-- ==========================================================
-- Create diagnosis cohort for MPS II patients
-- Cohort logic supports:
--   1) ≥2 specified diagnosis claims (E76.1)
--   2) OR incremental patients with unspecified dx (E76.3)
--      + evidence of treatment
-- Date window standardized across all diagnosis sources
-- ==========================================================

CREATE OR REPLACE TEMPORARY VIEW mpsii_diagnosis_table AS

-- ==========================================================
-- STEP 1: Capture patients with ≥1 SPECIFIED MPS II diagnosis
-- ICD-10: E76.1
-- Sources: Medical + Pharmacy claims
-- ==========================================================
WITH mpsii_1dx_specified AS (

    SELECT *
    FROM (

        -- ----------------------------------------------
        -- Medical claims with specified MPS II diagnosis
        -- ----------------------------------------------
        SELECT DISTINCT
            PATIENT_ID,
            COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
            SERVICE_DATE AS FILL_DATE,
            MEDICAL_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODES,
            KH_PLAN_ID AS KH_PLAN,
            PLACE_OF_SERVICE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E761%'

        UNION

        -- ----------------------------------------------
        -- Pharmacy claims with specified MPS II diagnosis
        -- Includes PAID transactions only
        -- ----------------------------------------------
        SELECT DISTINCT
            PATIENT_ID,
            PRESCRIBER_NPI AS NPI,
            FILL_DATE,
            PHARMACY_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
            COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
            NULL AS PLACE_OF_SERVICE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E761'
          AND TRANSACTION_STATUS = 'PAID'

    ) AS combined

    -- ----------------------------------------------
    -- Apply diagnosis observation window
    -- ----------------------------------------------
    WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
),

-- ==========================================================
-- STEP 2: Enforce ≥2 distinct diagnosis dates (specified)
-- Improves diagnostic certainty
-- ==========================================================
mpsii_2dx_specified AS (
    SELECT *
    FROM mpsii_1dx_specified
    WHERE patient_id IN (
        SELECT a.patient_id
        FROM mpsii_1dx_specified AS a
        GROUP BY a.patient_id
        HAVING COUNT(DISTINCT a.fill_date) >= 2
    )
),

-- ==========================================================
-- STEP 3: Capture patients with ≥1 UNSPECIFIED MPS diagnosis
-- ICD-10: E76.3
-- ==========================================================
mpsii_1dx_unspecified AS (

    SELECT *
    FROM (

        -- ----------------------------------------------
        -- Medical claims with unspecified MPS diagnosis
        -- ----------------------------------------------
        SELECT DISTINCT
            PATIENT_ID,
            COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
            SERVICE_DATE AS FILL_DATE,
            MEDICAL_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODES,
            KH_PLAN_ID AS KH_PLAN,
            PLACE_OF_SERVICE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E763%'

        UNION

        -- ----------------------------------------------
        -- Pharmacy claims with unspecified MPS diagnosis
        -- ----------------------------------------------
        SELECT DISTINCT
            PATIENT_ID,
            PRESCRIBER_NPI AS NPI,
            FILL_DATE,
            PHARMACY_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
            COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
            NULL AS PLACE_OF_SERVICE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E763'
          AND TRANSACTION_STATUS = 'PAID'

    ) AS combined

    -- ----------------------------------------------
    -- Apply diagnosis observation window
    -- ----------------------------------------------
    WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
),

-- ==========================================================
-- STEP 4: Require ≥2 diagnosis dates (unspecified)
-- ==========================================================
mpsii_2dx_unspecified AS (
    SELECT *
    FROM mpsii_1dx_unspecified
    WHERE patient_id IN (
        SELECT a.patient_id
        FROM mpsii_1dx_unspecified AS a
        GROUP BY a.patient_id
        HAVING COUNT(DISTINCT a.fill_date) >= 2
    )
),

-- ==========================================================
-- STEP 5: Identify treated patients among specified dx cohort
-- ==========================================================
mpsii_2dx_specified_tx AS (
    SELECT *
    FROM mpsii_treatment_table
    WHERE patient_id IN (
        SELECT DISTINCT patient_id
        FROM mpsii_2dx_specified
    )
),

-- ==========================================================
-- STEP 6: Incremental cohort logic
-- Include UNSPECIFIED dx patients ONLY IF:
--   1) They are treated (NDC / J-code evidence)
--   2) They are NOT already in specified dx + tx cohort
-- ==========================================================
incremental_patient AS (
    SELECT DISTINCT patient_id
    FROM mpsii_2dx_unspecified
    WHERE patient_id IN (
        SELECT DISTINCT patient_id
        FROM mpsii_treatment_table
        WHERE code IN ('54092070001','540920700','J1743')
    )
    AND patient_id NOT IN (
        SELECT DISTINCT patient_id
        FROM mpsii_2dx_specified_tx
    )
),

-- ==========================================================
-- STEP 7: Final diagnosis cohort
--   - All ≥2dx specified patients
--   - PLUS incremental unspecified + treated patients
-- ==========================================================
all_dx_patients_claims AS (
    SELECT *
    FROM mpsii_2dx_specified

    UNION

    SELECT *
    FROM mpsii_1dx_unspecified
    WHERE patient_id IN (
        SELECT patient_id
        FROM incremental_patient
    )
)

-- ==========================================================
-- Final output: Diagnosis-qualified patient claims
-- ==========================================================
SELECT *
FROM all_dx_patients_claims;


In [0]:
-- =====================================================
-- Create a temporary view to calculate diagnosed patients
-- at the HCO level (non-unique across institutions)
-- Diagnosis attribution is rolled up from HCP → HCO
-- =====================================================

CREATE OR REPLACE TEMPORARY VIEW dx_claims_non_unique AS

-- ---------------------------------------------
-- Step 1: Attach HCO NPI to diagnosis claims
-- Uses HCP → HCO affiliation reference
-- ---------------------------------------------
WITH t1 AS (
    SELECT
        a.*,          -- All diagnosis claim attributes
        b.hco_npi     -- Parent HCO mapped from HCP NPI
    FROM mpsii_diagnosis_table AS a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 AS b
        ON a.npi = b.hcp_npi
)

-- ---------------------------------------------
-- Step 2: Aggregate diagnosed patients at HCO level
-- Counts DISTINCT patients with qualifying diagnosis
-- ---------------------------------------------
SELECT
    hco_npi,
    COUNT(DISTINCT patient_id) AS dx_patients_non_unique
FROM t1

-- ---------------------------------------------
-- Filters:
-- 1) Exclude invalid / placeholder HCO NPIs
-- 2) Ensure provider attribution exists
-- ---------------------------------------------
WHERE hco_npi != '-'
  AND npi IS NOT NULL

-- ---------------------------------------------
-- Final aggregation & ranking
-- ---------------------------------------------
GROUP BY 1
ORDER BY 2 DESC;


In [0]:
-- ============================================================
-- Create UNIQUE treated patient counts at HCO level
-- Each patient is assigned to ONE primary treating HCP
-- HCP selection is based on:
--   1) Specialty priority
--   2) Number of visits
--   3) Most recent treatment date
-- ============================================================

CREATE OR REPLACE TEMPORARY VIEW tx_claims_unique AS

-- ============================================================
-- STEP 0: Base treatment population
-- ============================================================
WITH elaprase_treated_v1 AS (
    SELECT *
    FROM mpsii_treatment_table
),

-- ============================================================
-- STEP 1: Attach provider specialty & classify into buckets
-- Specialty hierarchy supports rare disease treatment logic
-- ============================================================
pulling_specialities AS (
    SELECT
        a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,

        -- --------------------------------------------
        -- Normalize specialties into analytical buckets
        -- --------------------------------------------
        CASE
            WHEN primary_specialty LIKE '%Genetic%'
              OR secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'

            WHEN primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'

            WHEN primary_specialty LIKE '%Psychiatry & Neurology%'
              OR secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR primary_specialty LIKE '%Neurological Surgery%'
              THEN 'Psychiatry & Neurology'

            WHEN primary_specialty LIKE '%Nurse Practitioner%'
              OR primary_specialty LIKE '%Physician Assistant%'
              THEN 'NPPA'

            WHEN primary_specialty LIKE '%Internal Medicine%'
              OR secondary_specialty LIKE '%Internal Medicine%'
              THEN 'PCP'

            WHEN primary_specialty LIKE '%Family Medicine%'
              OR secondary_specialty LIKE '%Family Medicine%'
              THEN 'PCP'

            WHEN a.npi IS NULL THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY

    FROM elaprase_treated_v1 a
    LEFT JOIN com_edp_prd.com_raw.kom_providers b
        ON a.npi = b.npi
),

-- ============================================================
-- STEP 2: Assign ONE primary treating HCP per patient
-- Ranking logic:
--   Priority (specialty) →
--   Visit frequency →
--   Recency →
--   Stable NPI tie-break
-- ============================================================
elaprase_treated_v2 AS (

    SELECT DISTINCT
        n_pats AS patient_id,
        npi
    FROM (

        SELECT DISTINCT
            NPI,
            SPECIALTY,
            PATIENT_ID AS N_PATS,
            FILL_DATE,
            KH_PLAN,
            HCO_PRIMARY_NPI,
            PLACE_OF_SERVICE,
            'TX' AS PATIENT_TYPE
        FROM (

            SELECT DISTINCT
                PATIENT_ID,
                FILL_DATE,
                KH_PLAN,
                HCO_PRIMARY_NPI,
                PLACE_OF_SERVICE,
                NPI,
                SPECIALTY,
                FINAL_HCP_RANK
            FROM (

                -- --------------------------------------------
                -- Collapse multiple ranking layers
                -- --------------------------------------------
                SELECT *,
                       DENSE_RANK() OVER (
                           PARTITION BY PATIENT_ID
                           ORDER BY HCP_RANK_1
                       ) AS FINAL_HCP_RANK
                FROM (

                    SELECT *,
                           MIN(HCP_RANK) OVER (
                               PARTITION BY PATIENT_ID, NPI
                           ) AS HCP_RANK_1
                    FROM (

                        -- --------------------------------------------
                        -- Core HCP ranking logic
                        -- --------------------------------------------
                        SELECT *,
                               RANK() OVER (
                                   PARTITION BY PATIENT_ID
                                   ORDER BY
                                       PRIORITY,
                                       NO_OF_VISITS DESC,
                                       DATE(FILL_DATE) DESC,
                                       NPI
                               ) AS HCP_RANK
                        FROM (

                            -- --------------------------------------------
                            -- Precompute visit counts & specialty priority
                            -- --------------------------------------------
                            SELECT
                                PATIENT_ID,
                                NPI,
                                SPECIALTY,
                                FILL_DATE,
                                KH_PLAN,
                                HCO_PRIMARY_NPI,
                                PLACE_OF_SERVICE,

                                -- Specialty priority (lower = higher priority)
                                CASE
                                    WHEN SPECIALTY = 'Geneticist' THEN 1
                                    WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2
                                    WHEN SPECIALTY = 'Pediatrician' THEN 3
                                    WHEN SPECIALTY = 'PCP' THEN 4
                                    WHEN SPECIALTY = 'NPPA' THEN 5
                                    WHEN SPECIALTY = 'Others' THEN 6
                                    ELSE 7
                                END AS PRIORITY,

                                COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                            FROM pulling_specialities
                            GROUP BY
                                PATIENT_ID,
                                NPI,
                                SPECIALTY,
                                FILL_DATE,
                                KH_PLAN,
                                HCO_PRIMARY_NPI,
                                PLACE_OF_SERVICE
                        )
                    )
                )
            )
            -- --------------------------------------------
            -- Retain only top-ranked HCP per patient
            -- --------------------------------------------
            WHERE HCP_RANK = 1
        )
    )
),

-- ============================================================
-- STEP 3: Filter valid HCP assignments
-- ============================================================
hcp_level AS (
    SELECT *
    FROM elaprase_treated_v2
    WHERE npi IS NOT NULL
),

-- ============================================================
-- STEP 4: Roll up unique patients from HCP → HCO
-- ============================================================
hco_level AS (
    SELECT
        a.*,
        b.hco_npi
    FROM hcp_level a
    LEFT JOIN cmpa_insights_internal_schema.reference_file_0109 b
        ON a.npi = b.hcp_npi
)

-- ============================================================
-- STEP 5: Final UNIQUE treated patient count by HCO
-- Each patient contributes to exactly ONE HCO
-- ============================================================
SELECT
    hco_npi,
    COUNT(DISTINCT patient_id) AS tx_patients_unique
FROM hco_level
WHERE hco_npi != '-'
GROUP BY 1
ORDER BY 2 DESC;


In [0]:
create or replace temporary view dx_claims_unique as 
with mpsii_diagnosis_v1 as (
  select *
from mpsii_diagnosis_table
),
pulling_specialities as (
  select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (select * from mpsii_diagnosis_v1) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi
),
mpsii_diagnosis_v2 as (
  select distinct n_pats as patient_id, npi 
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM pulling_specialities
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
),
hcp_level as (
    select * from mpsii_diagnosis_v2
where npi is not null
),
hco_level as (
    select a.*, b.hco_npi
    from hcp_level as a
    left join cmpa_insights_internal_schema.reference_file_0109 as b on a.npi = b.hcp_npi
)
select hco_npi, count(distinct patient_id) as dx_patients_unique
from hco_level
where hco_npi != '-'
group by 1 order by 2 desc

In [0]:
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical Events - Dx (COALESCE)
SELECT DISTINCT 
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Pharmacy Events - Dx (PRESCRIBER_NPI)
SELECT DISTINCT 
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 2: TREATMENT CLAIMS (5yr) - CORRECTED NPI LOGIC
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- Medical Events - NDC codes (COALESCE)
SELECT DISTINCT 
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Medical Events - Procedure codes (RENDERING_NPI only)
SELECT DISTINCT 
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743', 
                         'S9357', 'S9379', '38206', '38230', '38232', 
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'

UNION

-- Pharmacy Events - NDC codes (PRESCRIBER_NPI)
SELECT DISTINCT 
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 3: TREATMENT CLAIMS FOR ELIGIBILITY (2yr)
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';


-- =============================================================================
-- STEP 4: PATIENT ELIGIBILITY
-- =============================================================================

-- Specified: 2+ E761 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


-- Specified Patients: 2+ E761 Dx + any Tx in 2yr
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;


-- Incremental: 2+ E763 Dx dates (5yr)
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-11-30'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


-- Elaprase Tx in 2yr (for incremental eligibility)
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');


-- Incremental Patients: 2+ E763 Dx + Elaprase Tx in 2yr + NOT specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);


-- All Eligible Patients
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


-- =============================================================================
-- STEP 5: COMBINED Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

-- Dx claims (5yr)
SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
FROM all_dx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

UNION

-- Tx claims (5yr)
SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
FROM all_tx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);

In [0]:
-- =====================================================
-- Create non-unique patient counts at HCO level
-- Includes ALL eligible patients (Dx + Tx)
-- Patients may be counted across multiple HCOs
-- =====================================================

CREATE OR REPLACE TEMPORARY VIEW all_patients_non_unique AS

-- ---------------------------------------------
-- Step 1: HCP-level claims for eligible patients
-- ---------------------------------------------
WITH hcp_level AS (
    SELECT *
    FROM all_patient_claims
),

-- ---------------------------------------------
-- Step 2: Roll up provider attribution
-- Map HCP NPI → Parent HCO NPI
-- ---------------------------------------------
hco_level AS (
    SELECT
        a.*,            -- Patient-level claim attributes
        b.hco_npi       -- Parent HCO identifier
    FROM hcp_level AS a
    LEFT JOIN cmpa_insights_internal_schema.reference_file_0109 AS b
        ON a.npi = b.hcp_npi
)

-- ---------------------------------------------
-- Step 3: Aggregate patients at HCO level
-- Counts DISTINCT patients per HCO
-- ---------------------------------------------
SELECT
    hco_npi,
    COUNT(DISTINCT patient_id) AS all_patients_non_unique
FROM hco_level

-- ---------------------------------------------
-- Filters:
--   1) Ensure provider attribution exists
--   2) Exclude invalid / placeholder HCO NPIs
-- ---------------------------------------------
WHERE npi IS NOT NULL
  AND hco_npi != '-'

-- ---------------------------------------------
-- Final grouping & ranking
-- ---------------------------------------------
GROUP BY 1
ORDER BY 2 DESC;


In [0]:
-- ============================================================
-- Create UNIQUE eligible patient counts at HCO level
-- Each patient is assigned to ONE primary HCP and ONE HCO
-- Attribution logic uses combined Dx + Tx activity
-- ============================================================

CREATE OR REPLACE TEMPORARY VIEW all_patients_unique AS

-- ============================================================
-- STEP 1: Build HCP-level metrics per patient
-- Metrics combine Dx + Tx claims for eligibility population
-- ============================================================
WITH hcp_metrics AS (
    SELECT 
        a.PATIENT_ID,
        a.NPI,

        -- ----------------------------------------------------
        -- Specialty classification (normalized buckets)
        -- ----------------------------------------------------
        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 'Geneticist'

            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 'Psychiatry & Neurology'

            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 'Pediatrician'

            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 'PCP'

            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 'NPPA'

            WHEN a.NPI IS NULL 
                THEN 'NA'

            ELSE 'Others'
        END AS SPECIALTY,

        -- ----------------------------------------------------
        -- Specialty priority (Tier 1 ranking)
        -- Lower number = higher attribution priority
        -- ----------------------------------------------------
        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' 
              OR p.secondary_specialty LIKE '%Genetic%' 
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
              OR p.primary_specialty LIKE '%Neurological Surgery%' 
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%' 
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%' 
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%' 
              OR p.secondary_specialty LIKE '%Family Medicine%' 
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
              OR p.primary_specialty LIKE '%Physician Assistant%' 
                THEN 5
            WHEN a.NPI IS NULL 
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        -- ----------------------------------------------------
        -- Visit metrics (Tier 2)
        -- Distinct service dates across Dx + Tx
        -- ----------------------------------------------------
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        -- Dx-only visit count (reference metric)
        COUNT(DISTINCT CASE 
            WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE 
        END) AS DX_VISITS,

        -- Tx-only visit count (reference metric)
        COUNT(DISTINCT CASE 
            WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE 
        END) AS TX_VISITS,

        -- ----------------------------------------------------
        -- Most recent interaction date (Tier 3)
        -- ----------------------------------------------------
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p 
        ON a.NPI = p.NPI

    GROUP BY 
        a.PATIENT_ID, 
        a.NPI,
        p.primary_specialty, 
        p.secondary_specialty
),

-- ============================================================
-- STEP 2: Rank HCPs per patient
-- Ranking hierarchy:
--   Tier 1: Specialty priority
--   Tier 2: Total visits
--   Tier 3: Recency
--   Tier 4: NPI tie-breaker
-- ============================================================
ranked_hcps AS (
    SELECT 
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID 
            ORDER BY 
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
),

-- ============================================================
-- STEP 3: Select primary HCP per patient
-- ============================================================
hcp_level AS (
    SELECT 
        PATIENT_ID,
        NPI AS PRIMARY_HCP_NPI,
        SPECIALTY AS PRIMARY_HCP_SPECIALTY,
        SPECIALTY_PRIORITY,
        NO_OF_VISITS,
        DX_VISITS,
        TX_VISITS,
        MOST_RECENT_VISIT,
        HCP_RANK
    FROM ranked_hcps
    WHERE HCP_RANK = 1
),

-- ============================================================
-- STEP 4: Roll up primary HCP → primary HCO
-- ============================================================
hco_level AS (
    SELECT 
        a.*,
        b.hco_npi
    FROM hcp_level AS a
    LEFT JOIN cmpa_insights_internal_schema.reference_file_0109 AS b
        ON a.primary_hcp_npi = b.hcp_npi
)

-- ============================================================
-- STEP 5: Final UNIQUE patient count by HCO
-- Each patient contributes to exactly ONE HCO
-- ============================================================
SELECT
    hco_npi,
    COUNT(DISTINCT patient_id) AS all_patients_unique
FROM hco_level
WHERE primary_hcp_npi IS NOT NULL
  AND hco_npi != '-'
GROUP BY 1
ORDER BY 2 DESC;

-- Debug option:
-- SELECT * FROM hcp_level;


### HCO Level View

In [0]:
-- =============================================================================
-- BASE TABLE: Target HCO Universe
-- Purpose:
--   Define the fixed list of HCO NPIs to be analyzed (Tier1 HCOs)
-- =============================================================================
WITH base_table (hco_npi) AS (
  VALUES
    ('1467525790'), ('1023105400'), ('1649347469'), ('1144266024'),
    ('1003878539'), ('1760476659'), ('1578693321'), ('1396882205'),
    ('1639370059'), ('1326092404'), ('1295789907'), ('1013143213'),
    ('1912939703'), ('1346297843'), ('1235234535'), ('1053632463'),
    ('1376544320'), ('1144548322'), ('1750482022'), ('1598784555'),
    ('1659877280'), ('1265694442'), ('1013924182'), ('1093808040'),
    ('1285174649'), ('1114969169'), ('1932280666'), ('1063702785'),
    ('1164426896'), ('1669683512'), ('1093894131'), ('1215921457'),
    ('1548212988'), ('1437365186'), ('1184649345'), ('1669462420'),
    ('1003063280'), ('1114924834'), ('1013924372'), ('1225249865'),
    ('1043447253'), ('1003961251'), ('1073053757'), ('1366556227'),
    ('1285832634'), ('1609824010'), ('1023188851'), ('1083789630'),
    ('1336495910'), ('1891765178'), ('1033439732'), ('1700128592'),
    ('1194787218'), ('1235582925'), ('1477643690'), ('1104819366'),
    ('1477549756'), ('1205935012'), ('1235214834'), ('1568596765'),
    ('1669429577'), ('1275564098'), ('1649261462'), ('1235339227'),
    ('1750458485'), ('1275694184'), ('1154302727'), ('1003102781'),
    ('1083949382'), ('1164686879'), ('1013062769'), ('1336245828'),
    ('1235148594'), ('1689747552'), ('1760480503'), ('1366515488'),
    ('1083630073'), ('1679973364')
),

/* =============================================================================
   STEP 15: Pull latest VALID HCO address from VOD
   Logic:
     - Join VOD HCO → VOD Address
     - Retain only VALID, ACTIVE, VERIFIED addresses
     - Rank by most recent modified date
============================================================================= */
hco_zip_v1 AS (
    SELECT DISTINCT
        a.npi_num__v AS hco_npi,
        b.address_line_1__v AS hco_address,
        b.postal_code_cda__v AS hco_postal_code,
        b.modified_date__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.npi_num__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_raw.vod_hco a
    JOIN com_raw.vod_address b
        ON b.entity_vid__v = a.vid__v
       AND b.entity_type__v = 'HCO'
       AND b.record_state__v = 'VALID'
       AND b.address_status__v IN ('A','DS')
       AND b.address_verification_status__v NOT IN ('NS','U')
    WHERE a.npi_num__v IN (
        SELECT DISTINCT hco_npi
        FROM base_table
    )
),

/* =============================================================================
   STEP 16: Select most recent address per HCO
============================================================================= */
hco_zip_v2 AS (
    SELECT
        hco_npi,
        hco_address,
        hco_postal_code
    FROM hco_zip_v1
    WHERE rn = 1
),

/* =============================================================================
   STEP 17: ZIP backfill using Komodo providers
   Logic:
     - Prefer VOD ZIP when available
     - Backfill from Komodo ORGANIZATION-level provider records
     - Default ZIP to '-' if still missing
============================================================================= */
pulling_hco_zip_using_vod_komodo AS (
    SELECT
        b.hco_npi,
        CASE 
            WHEN v.hco_postal_code IS NOT NULL THEN v.hco_address
            ELSE k.provider_address
        END AS hco_address,
        COALESCE(v.hco_postal_code, k.provider_zip, '-') AS hco_zip
    FROM base_table b
    LEFT JOIN hco_zip_v2 v
        ON b.hco_npi = v.hco_npi
    LEFT JOIN com_raw.kom_providers k
        ON b.hco_npi = k.npi
       AND k.provider_type = 'ORGANIZATION'
),

/* =============================================================================
   STEP 18: Territory & Region assignment
   Logic:
     - Map ZIP → territory and region
     - Handle missing ZIPs safely using TRY_CAST
============================================================================= */
territory_region_reassignment AS (
    SELECT
        a.hco_npi,
        a.hco_address,
        a.hco_zip,
        m.territory_id,
        m.territory_name AS territory,
        m.region_id,
        m.region_name AS region
    FROM pulling_hco_zip_using_vod_komodo a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping m
        ON TRY_CAST(NULLIF(a.hco_zip, '-') AS BIGINT) = m.zipcode
),

/* =============================================================================
   STEP 19: Attach patient volume metrics
   Metrics included:
     - Tx patients (unique & non-unique)
     - Dx patients (unique & non-unique)
     - All eligible patients (unique & non-unique)
============================================================================= */
pulling_relevant_counts AS (
    SELECT
        a.*,
        b.tx_patients_non_unique,
        c.dx_patients_non_unique,
        d.tx_patients_unique,
        e.dx_patients_unique,
        f.all_patients_non_unique,
        g.all_patients_unique
    FROM territory_region_reassignment a
    LEFT JOIN tx_claims_non_unique b
        ON a.hco_npi = b.hco_npi
    LEFT JOIN dx_claims_non_unique c
        ON a.hco_npi = c.hco_npi
    LEFT JOIN tx_claims_unique d
        ON a.hco_npi = d.hco_npi
    LEFT JOIN dx_claims_unique e
        ON a.hco_npi = e.hco_npi
    LEFT JOIN all_patients_non_unique f
        ON a.hco_npi = f.hco_npi
    LEFT JOIN all_patients_unique g
        ON a.hco_npi = g.hco_npi
)

-- =============================================================================
-- FINAL OUTPUT:
-- One row per HCO with:
--   - Address & geography
--   - Territory & region
--   - Dx / Tx / All patient volumes (unique & non-unique)
-- =============================================================================
SELECT *
FROM pulling_relevant_counts;


In [0]:
SELECT
    hco_npi,
    LISTAGG(DISTINCT hcp_npi, ',') 
        WITHIN GROUP (ORDER BY hcp_npi) AS hcp_npi_list
FROM cmpa_insights_internal_schema.reference_file_0109
WHERE hcp_npi <> '-'
  AND hco_npi IN (
    '1467525790','1023105400','1649347469','1144266024','1003878539',
    '1760476659','1578693321','1396882205','1639370059','1326092404',
    '1295789907','1013143213','1912939703','1346297843','1235234535',
    '1053632463','1376544320','1144548322','1750482022','1598784555',
    '1659877280','1265694442','1013924182','1093808040','1285174649',
    '1114969169','1932280666','1063702785','1164426896','1669683512',
    '1093894131','1215921457','1548212988','1437365186','1184649345',
    '1669462420','1003063280','1114924834','1013924372','1225249865',
    '1043447253','1003961251','1073053757','1366556227','1285832634',
    '1609824010','1023188851','1083789630','1336495910','1891765178',
    '1033439732','1700128592','1194787218','1235582925','1477643690',
    '1104819366','1477549756','1205935012','1235214834','1568596765',
    '1669429577','1275564098','1649261462','1235339227','1750458485',
    '1275694184','1154302727','1003102781','1083949382','1164686879',
    '1013062769','1336245828','1235148594','1689747552','1760480503',
    '1366515488','1083630073','1679973364'
) and hcp_npi in (select distinct npi from mpsii_treatment_table where npi is not null)
GROUP BY hco_npi;


### Tier 1 HCO Level Additional Columns

In [0]:
select distinct npi_num__v as hco_npi, corporate_name__v as hco_name_opendata
from com_edp_prd.com_raw.vod_hco
where npi_num__v in (
  '1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364',
  '1093728743', '1184722779', '1861439952'
)

In [0]:
select distinct NPI as hco_npi, ORGANIZATION_NAME as hco_name
from com_edp_prd.com_raw.kom_providers
where PROVIDER_TYPE = 'ORGANIZATION' and npi in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952')

In [0]:
with base AS (
  SELECT DISTINCT
      a.npi_num__v        AS hco_npi,
      a.corporate_name__v AS hco_name,

      -- Hospital parent (lowest parent level in this rollup)
      b.npi_num__v        AS hospital_parent_npi,
      b.corporate_name__v AS hospital_parent_name,

      -- Immediate parent (mid-level)
      c.npi_num__v        AS immediate_parent_npi,
      c.corporate_name__v AS immediate_parent_name,

      -- Top parent (highest-level rollup)
      d.npi_num__v        AS top_parent_npi,
      d.corporate_name__v AS top_parent_name

  FROM com_raw.vod_hco a
  LEFT JOIN com_raw.vod_hco b
      ON a.hospital_parent__v = b.vid__v
  LEFT JOIN com_raw.vod_hco c
      ON a.immediate_parent__v = c.vid__v
  LEFT JOIN com_raw.vod_hco d
      ON a.top_parent__v = d.vid__v

  WHERE a.npi_num__v IN (
      '1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952'
  )
),

/* ---------------------------------------------------------------------------
   STEP 2: Resolve a single parent NPI/name per HCO (best available parent)
   - Preference order:
       1) top parent
       2) immediate parent
       3) hospital parent
       4) null (no parent found)
   - Join the resolved parent fields back onto the hco_360 rows
   --------------------------------------------------------------------------- */
parent_mapping AS (
  SELECT DISTINCT
          hco_npi,
          hco_name,

          /* Parent NPI + Name resolved from SAME LEVEL */
          CASE
              WHEN top_parent_npi IS NOT NULL THEN top_parent_npi
              WHEN immediate_parent_npi IS NOT NULL THEN immediate_parent_npi
              WHEN hospital_parent_npi IS NOT NULL THEN hospital_parent_npi
              ELSE NULL
          END AS parent_npi,

          CASE
              WHEN top_parent_npi IS NOT NULL THEN top_parent_name
              WHEN immediate_parent_npi IS NOT NULL THEN immediate_parent_name
              WHEN hospital_parent_npi IS NOT NULL THEN hospital_parent_name
              ELSE NULL
          END AS parent_name
      FROM base
)

select * from parent_mapping

In [0]:
select distinct npi_num__v, top_parent__v, immediate_parent__v, hospital_parent__v
from com_raw.vod_hco
where npi_num__v in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952') and (top_parent__v is not null or hospital_parent__v is not null or immediate_parent__v is not null)

In [0]:
with t1 as (select distinct npi_num__v, top_parent__v, immediate_parent__v, hospital_parent__v
from com_raw.vod_hco
where npi_num__v in ('1467525790', '1023105400', '1649347469', '1144266024', '1003878539', '1760476659', '1578693321', '1396882205', '1639370059', '1326092404', '1295789907', '1013143213', '1912939703', '1346297843', '1235234535', '1053632463', '1376544320', '1144548322', '1750482022', '1598784555', '1659877280', '1265694442', '1013924182', '1093808040', '1285174649', '1114969169', '1932280666', '1063702785', '1164426896', '1669683512', '1093894131', '1215921457', '1548212988', '1437365186', '1184649345', '1669462420', '1003063280', '1114924834', '1013924372', '1225249865', '1043447253', '1003961251', '1073053757', '1366556227', '1285832634', '1609824010', '1023188851', '1083789630', '1336495910', '1891765178', '1033439732', '1700128592', '1194787218', '1235582925', '1477643690', '1104819366', '1477549756', '1205935012', '1235214834', '1568596765', '1669429577', '1275564098', '1649261462', '1235339227', '1750458485', '1275694184', '1154302727', '1003102781', '1083949382', '1164686879', '1013062769', '1336245828', '1235148594', '1689747552', '1760480503', '1366515488', '1083630073', '1679973364', '1093728743', '1184722779', '1861439952') and (top_parent__v is not null or hospital_parent__v is not null or immediate_parent__v is not null))
select a.npi_num__v as hco_npi, e.corporate_name__v as hco_name, a.top_parent__v, b.corporate_name__v as top_parent_name,
a.immediate_parent__v, c.corporate_name__v as immediate_parent_name,
a.hospital_parent__v, d.corporate_name__v as hospital_parent_name
from t1 as a
left join com_raw.vod_hco as b on a.top_parent__v = b.vid__v
left join com_raw.vod_hco as c on a.immediate_parent__v = c.vid__v
left join com_raw.vod_hco as d on a.hospital_parent__v = d.vid__v
left join com_raw.vod_hco as e on a.npi_num__v = e.npi_num__v

In [0]:
WITH t1 AS (
    SELECT DISTINCT 
        npi_num__v, 
        vid__v,
        top_parent__v, 
        immediate_parent__v, 
        hospital_parent__v 
    FROM com_raw.vod_hco 
    WHERE 
        top_parent__v = '931278329388861343'
        OR immediate_parent__v = '931278329388861343'
        OR hospital_parent__v = '931278329388861343'
)
SELECT 
    a.vid__v as child_vid,
    a.npi_num__v AS child_hco_npi,
    e.corporate_name__v AS child_hco_name,
    a.top_parent__v,
    b.corporate_name__v AS top_parent_name,
    a.immediate_parent__v,
    c.corporate_name__v AS immediate_parent_name,
    a.hospital_parent__v,
    d.corporate_name__v AS hospital_parent_name
FROM t1 AS a
LEFT JOIN com_raw.vod_hco AS b ON a.top_parent__v = b.vid__v
LEFT JOIN com_raw.vod_hco AS c ON a.immediate_parent__v = c.vid__v
LEFT JOIN com_raw.vod_hco AS d ON a.hospital_parent__v = d.vid__v
LEFT JOIN com_raw.vod_hco AS e ON a.npi_num__v = e.npi_num__v

### Parent Level Info (16th Jan)

In [0]:
WITH target_top_parents AS (
    SELECT DISTINCT top_parent__v
    FROM com_raw.vod_hco
    WHERE npi_num__v IN ('1467525790','1023105400','1649347469','1144266024','1003878539','1760476659','1578693321','1396882205','1639370059','1326092404','1295789907','1013143213','1912939703','1346297843','1235234535','1053632463','1376544320','1144548322','1750482022','1598784555','1659877280','1265694442','1013924182','1093808040','1285174649','1114969169','1932280666','1063702785','1164426896','1669683512','1093894131','1215921457','1548212988','1437365186','1184649345','1669462420','1003063280','1114924834','1013924372','1225249865','1043447253','1003961251','1073053757','1366556227','1285832634','1609824010','1023188851','1083789630','1336495910','1891765178','1033439732','1700128592','1194787218','1235582925','1477643690','1104819366','1477549756','1205935012','1235214834','1568596765','1669429577','1275564098','1649261462','1235339227','1750458485','1275694184','1154302727','1003102781','1083949382','1164686879','1013062769','1336245828','1235148594','1689747552','1760480503','1366515488','1083630073','1679973364')
      AND top_parent__v IS NOT NULL
),

top_parent_child AS (
    SELECT
        tp.top_parent__v AS top_parent_vid,
        p.corporate_name__v AS top_parent_name,
        p.npi_num__v AS top_parent_npi,
        e.name AS top_parent_type,
        c.vid__v AS child_vid,
        c.corporate_name__v AS child_name,
        c.npi_num__v AS child_npi,
        d.name AS child_type,
        c.hco_status__v AS child_status
    FROM target_top_parents tp
    JOIN com_raw.vod_hco p
        ON tp.top_parent__v = p.vid__v
    JOIN com_raw.vod_hco c
        ON (tp.top_parent__v = c.top_parent__v
         OR tp.top_parent__v = c.immediate_parent__v
         OR tp.top_parent__v = c.hospital_parent__v)
         and c.npi_num__v in (select distinct hco_npi_old from com_edp_prd.cmpa_insights_internal_schema.reference_file)
    LEFT JOIN com_raw.vod_references d
        ON c.hco_type__v = d.code AND d.reference_type = 'HCOType'
    LEFT JOIN com_raw.vod_references e
        ON p.hco_type__v = e.code AND e.reference_type = 'HCOType'
    ORDER BY top_parent_vid, child_vid
),

appending_tier_flag AS (
  SELECT *,
    CASE
      WHEN child_npi IN ('1467525790','1023105400','1649347469','1144266024','1003878539','1760476659','1578693321','1396882205','1639370059','1326092404','1295789907','1013143213','1912939703','1346297843','1235234535','1053632463','1376544320','1144548322','1750482022','1598784555','1659877280','1265694442','1013924182','1093808040','1285174649','1114969169','1932280666','1063702785','1164426896','1669683512','1093894131','1215921457','1548212988','1437365186','1184649345','1669462420','1003063280','1114924834','1013924372','1225249865','1043447253','1003961251','1073053757','1366556227','1285832634','1609824010','1023188851','1083789630','1336495910','1891765178','1033439732','1700128592','1194787218','1235582925','1477643690','1104819366','1477549756','1205935012','1235214834','1568596765','1669429577','1275564098','1649261462','1235339227','1750458485','1275694184','1154302727','1003102781','1083949382','1164686879','1013062769','1336245828','1235148594','1689747552','1760480503','1366515488','1083630073','1679973364')
      THEN 1 ELSE 0
    END AS tier1_flag
  FROM top_parent_child
),

child_count AS (
    SELECT top_parent_vid, COUNT(DISTINCT child_vid) AS no_child_accounts
    FROM appending_tier_flag
    GROUP BY top_parent_vid
),

appending_child_count AS (
  SELECT a.*, b.no_child_accounts
  FROM appending_tier_flag a
  LEFT JOIN child_count b
    ON a.top_parent_vid = b.top_parent_vid
),

/* -------------------------------------------------------------------------
   NEW: Tier1-only base_table built from your result (no hard-coded VALUES)
   ------------------------------------------------------------------------- */
base_table AS (
  SELECT DISTINCT TRY_CAST(child_npi AS STRING) AS hco_npi
  FROM appending_child_count
  WHERE tier1_flag = 1
    AND child_npi IS NOT NULL
),

hco_engaged AS (
  SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
  FROM com_edp_prd.com_raw.vcrm_call2__v a
  LEFT JOIN com_intgr.customer b
    ON a.account__v = b.id
  WHERE TRY_CAST(b.npi__v AS STRING) IN (SELECT hco_npi FROM base_table)
    AND a.call_date__v BETWEEN '2025-07-01' AND '2025-12-31'
),

hco_profiled AS (
  SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
  FROM com_intgr.survey_target a
  LEFT JOIN com_intgr.customer b
    ON a.account__v = b.id
  WHERE TRY_CAST(b.npi__v AS STRING) IN (SELECT hco_npi FROM base_table)
),

tier1_flags AS (
  SELECT
    bt.hco_npi,
    CASE WHEN he.npi IS NOT NULL THEN 1 ELSE 0 END AS is_hco_engaged,
    CASE WHEN hp.npi IS NOT NULL THEN 1 ELSE 0 END AS is_hco_profiled
  FROM base_table bt
  LEFT JOIN hco_engaged he
    ON bt.hco_npi = he.npi
  LEFT JOIN hco_profiled hp
    ON bt.hco_npi = hp.npi
),

/* ------------------- your existing address/territory pipeline ------------------- */
target_vids AS (
    SELECT DISTINCT child_vid AS hco_vid
    FROM appending_child_count
    WHERE child_vid IS NOT NULL
    UNION
    SELECT DISTINCT top_parent_vid AS hco_vid
    FROM appending_child_count
    WHERE top_parent_vid IS NOT NULL
),

hco_address_latest AS (
    SELECT
        b.entity_vid__v AS hco_vid,
        b.address_line_1__v AS address_line_1,
        b.postal_code_cda__v AS postal_code,
        b.modified_date__v,
        ROW_NUMBER() OVER (
            PARTITION BY b.entity_vid__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_raw.vod_address b
    JOIN target_vids t
        ON t.hco_vid = b.entity_vid__v
    WHERE b.entity_type__v = 'HCO'
      AND b.record_state__v = 'VALID'
      AND b.address_status__v IN ('A','DS')
      AND b.address_verification_status__v NOT IN ('NS','U')
),

hco_address_latest_1 AS (
    SELECT hco_vid, address_line_1, postal_code
    FROM hco_address_latest
    WHERE rn = 1
),

addresses_child_top_parent AS (
    SELECT
        a.*,
        caddr.address_line_1 AS child_address,
        caddr.postal_code    AS child_zip,
        paddr.address_line_1 AS top_parent_address,
        paddr.postal_code    AS top_parent_zip
    FROM appending_child_count a
    LEFT JOIN hco_address_latest_1 caddr
        ON a.child_vid = caddr.hco_vid
    LEFT JOIN hco_address_latest_1 paddr
        ON a.top_parent_vid = paddr.hco_vid
),

territory_region_child AS (
    SELECT
        a.*,
        COALESCE(z.territory_name, '-') AS child_territory,
        COALESCE(z.region_name, '-')    AS child_region,
        COALESCE(z.city, '-')           AS child_city,
        COALESCE(z.state, '-')          AS child_state
    FROM addresses_child_top_parent a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON TRY_CAST(NULLIF(a.child_zip, '-') AS BIGINT) = z.zipcode
),

territory_region_parent AS (
    SELECT
        a.*,
        COALESCE(z.territory_name, '-') AS top_parent_territory,
        COALESCE(z.region_name, '-')    AS top_parent_region,
        COALESCE(z.city, '-')           AS top_parent_city,
        COALESCE(z.state, '-')          AS top_parent_state
    FROM territory_region_child a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON TRY_CAST(NULLIF(a.top_parent_zip, '-') AS BIGINT) = z.zipcode
),

/* -------------------------------------------------------------------------
   FINAL: attach engaged/profiled to CHILD NPI; NA when tier1_flag = 0
   ------------------------------------------------------------------------- */
final as (SELECT
  trp.*,

  CASE
    WHEN trp.tier1_flag = 0 THEN 'Not applicable'
    WHEN tf.is_hco_engaged = 1 THEN '1'
    ELSE '0'
  END AS is_hco_engaged,

  CASE
    WHEN trp.tier1_flag = 0 THEN 'Not applicable'
    WHEN tf.is_hco_profiled = 1 THEN '1'
    ELSE '0'
  END AS is_hco_profiled

FROM territory_region_parent trp
LEFT JOIN tier1_flags tf
  ON TRY_CAST(trp.child_npi AS STRING) = tf.hco_npi)

select * from final

In [0]:
WITH base_npis AS (
    select distinct hco_npi_old as npi
    from cmpa_insights_internal_schema.reference_file
),

hco_engaged AS (
    SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
    FROM com_edp_prd.com_raw.vcrm_call2__v a
    JOIN com_intgr.customer b
        ON a.account__v = b.id
    WHERE a.call_date__v BETWEEN '2025-07-01' AND '2025-12-31'
),

hco_profiled AS (
    SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
    FROM com_intgr.survey_target a
    JOIN com_intgr.customer b
        ON a.account__v = b.id
)

SELECT
    b.npi,
    CASE WHEN e.npi IS NOT NULL THEN 1 ELSE 0 END AS is_engaged,
    CASE WHEN p.npi IS NOT NULL THEN 1 ELSE 0 END AS is_profiled
FROM base_npis b
LEFT JOIN hco_engaged e
    ON b.npi = e.npi
LEFT JOIN hco_profiled p
    ON b.npi = p.npi
ORDER BY b.npi;


In [0]:
SELECT DISTINCT
    b.npi__v AS npi, a.entity_display_name__v as name_from_crm_calls_table
  FROM com_edp_prd.com_raw.vcrm_call2__v AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  and a.call_date__v between '2025-07-01' and '2025-12-31'
  where b.npi__v is not null

In [0]:
with hco_zip_v1 AS (
    SELECT DISTINCT
        a.npi_num__v AS hco_npi_active,
        b.address_line_1__v as hco_address,
        b.postal_code_cda__v AS hco_postal_code,
        b.modified_date__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.npi_num__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_raw.vod_hco a
    JOIN com_raw.vod_address b
        ON b.entity_vid__v = a.vid__v
       AND b.entity_type__v = 'HCO'
       AND b.record_state__v = 'VALID'
       AND b.address_status__v IN ('A','DS')
       AND b.address_verification_status__v NOT IN ('NS','U')
    WHERE a.npi_num__v IN (
        '1548330350', '1891126488', '1598103434', '1801910492', '1821004151', '1073558912', '1154369114', '1164448486', '1861415903', '1003381211'
    )
),
target_hco_npis AS (
    SELECT '1548330350' AS hco_npi_active UNION ALL
    SELECT '1891126488' UNION ALL
    SELECT '1598103434' UNION ALL
    SELECT '1801910492' UNION ALL
    SELECT '1821004151' UNION ALL
    SELECT '1073558912' UNION ALL
    SELECT '1154369114' UNION ALL
    SELECT '1164448486' UNION ALL
    SELECT '1861415903' UNION ALL
    SELECT '1003381211'
),


/* ---------------------------------------------------------------------------
   STEP 15: Select latest ZIP per HCO (rn=1)
   --------------------------------------------------------------------------- */
hco_zip_v2 AS (
    SELECT
        hco_npi_active,
        hco_address,
        hco_postal_code
    FROM hco_zip_v1
    WHERE rn = 1
),

/* ---------------------------------------------------------------------------
   STEP 16: Enrich / backfill HCO ZIP using:
   1) VOD latest ZIP (preferred)
   2) Komodo provider_zip (fallback)
   --------------------------------------------------------------------------- */
pulling_hco_zip_using_vod_komodo AS (
    SELECT
        a.* ,
        case when b.hco_postal_code is not null then b.hco_address else c.provider_address end as hco_address,
        COALESCE(b.hco_postal_code, c.provider_zip, '-') AS hco_zip
    FROM target_hco_npis a
    LEFT JOIN hco_zip_v2 b
        ON a.hco_npi_active = b.hco_npi_active
    LEFT JOIN com_raw.kom_providers c
        ON a.hco_npi_active = c.npi
       AND c.provider_type = 'ORGANIZATION'
),

/* ---------------------------------------------------------------------------
   STEP 17: Reassign territory & region using ZIP
   - Uses HCO ZIP if present; otherwise falls back to HCP ZIP
   --------------------------------------------------------------------------- */
territory_region_reassignment AS (
    SELECT
        a.* ,
        b.territory_id,
        b.territory_name AS territory,
        b.region_id,
        b.region_name AS region,
        city,
        state
    FROM pulling_hco_zip_using_vod_komodo a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b
        ON COALESCE(
               TRY_CAST(NULLIF(a.hco_zip, '-') AS BIGINT),
               TRY_CAST(NULLIF(a.hco_zip, '-') AS BIGINT)
           ) = b.zipcode
)

select a.*, corporate_name__v from territory_region_reassignment as a
left join com_raw.vod_hco on hco_npi_active = npi_num__v